<!-- nav -->
[← 11 · Constraints over collections](11_Constraints_Over_Collections.ipynb) · [Home](../../README.md) · [13 · Analyzing and rewriting constraints →](13_Analyzing_And_Rewriting_Constraints.ipynb)

# 12 · Deciding what you can now

**The problem.** Two questions keep coming back. A new product arrives with its own limits (a frozen vaccine at −25 to
−15 °C, for up to 120 hours) and the constraint must be written again, slightly differently. And while a shipment is
still on the road, "will it arrive in time?" depends on a number nobody has yet: how many more hours the trip takes.

**Partial evaluation** answers both. Give it the variables you know, and it evaluates everything they decide, leaving
the rest as an expression: the **residual**, the question that's left.

In [1]:
from mbse.Expressions.Patterns.Expressions import Simplifiers, Expressions as E, Partials
from mbse.Expressions.Framework import Symbolics
from toolkit import cold_chain, show

store = cold_chain()
Expression = store.Expressions.Framework.Expression
shipment, low, high, max_hours, to_go = (E.variable(name) for name in ("shipment", "low", "high", "max_hours", "to_go"))
celsius = shipment.celsius
with store.transaction():
    template = Expression(lambda e: e.root(
        low.le(celsius).and_(celsius.le(high)).and_(shipment.hours.add(to_go).le(max_hours))))
print(show(template))
sorted(Symbolics.free(template, E.DIALECT))

and(and(le(low, get(shipment, 'celsius')), le(get(shipment, 'celsius'), high)), le(add(get(shipment, 'hours'), to_go), max_hours))


['high', 'low', 'max_hours', 'shipment', 'to_go']

`Symbolics.free(constraint, dialect)` gives the names a constraint needs from its scope: the shipment, the product's
limits, and the hours to go.

<a id="T12.1"></a>
## 12.1 · One template, many products

Binding the limits gives each product its constraint. `Partials.OfAny(constraint, variables)` gives the residual as a
writer, which is written where it's wanted, here as the root of a new expression:

In [2]:
with store.transaction():
    vaccine = Expression(lambda e: e.root(Partials.OfAny(template, {"low": 2.0, "high": 8.0, "max_hours": 72})))
    frozen = Expression(lambda e: e.root(Partials.OfAny(template, {"low": -25.0, "high": -15.0, "max_hours": 120})))
for constraint in (vaccine, frozen):
    print(show(constraint), sorted(Symbolics.free(constraint, E.DIALECT)))

and(and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0)), le(add(get(shipment, 'hours'), to_go), 72)) ['shipment', 'to_go']
and(and(le(-25.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), -15.0)), le(add(get(shipment, 'hours'), to_go), 120)) ['shipment', 'to_go']


The residuals are ordinary expressions: they save, translate and evaluate like any other, and don't refer back to
the template.

<a id="T12.2"></a>
## 12.2 · What the shipment decides

Now bind a shipment, but not the hours to go. Whatever the shipment decides is evaluated, and Kleene's logic (case
study 10) simplifies what's left: `and(true, x)` is `x`, and `and(false, x)` is false whatever `x` is.

In [3]:
with store.transaction():
    s1 = store.Shipment(lambda s: s.id("S1").celsius(5.5).hours(30))
    s2 = store.Shipment(lambda s: s.id("S2").celsius(9.1).hours(30))
    s3 = store.Shipment(lambda s: s.id("S3").hours(20))  # no temperature yet
with store.transaction():
    residuals = {s.id.get(): Expression(lambda e, s=s: e.root(Partials.OfAny(vaccine, {"shipment": s})))
                 for s in (s1, s2, s3)}
for name, residual in residuals.items():
    print(name, show(residual))

S1 le(add(30, to_go), 72)
S2 False
S3 and(and(le(2.0, get(shipment, 'celsius')), le(get(shipment, 'celsius'), 8.0)), le(add(20, to_go), 72))


- S1 is in range, so all that's left is the clock: it arrives in time if 30 more hours, plus what's to go, stay
  within 72.
- S2 is out of range, so it fails whatever the trip takes.
- S3 has no temperature, so its range check stays as it was, still about `shipment`, and the clock check is reduced.

<a id="T12.3"></a>
## 12.3 · The residual agrees with the constraint

A residual is only useful if it means the same thing. Finishing it with the rest of the variables gives what the
original gives with all of them:

In [4]:
for s in (s1, s2, s3):
    residual = residuals[s.id.get()]
    for hours in (10, 50):
        whole = Simplifiers.OfAny(vaccine, {"shipment": s, "to_go": hours}).evaluate()
        assert Simplifiers.OfAny(residual, {"shipment": s, "to_go": hours}).evaluate() == whole
        print(s.id.get(), hours, whole)

S1 10 True
S1 50 False
S2 10 False
S2 50 False
S3 10 None
S3 50 None


<a id="T12.4"></a>
## 12.4 · Lets of known values

A let whose value is known binds it in its body, and disappears once nothing refers to it:

In [5]:
with store.transaction():
    total = Expression(lambda e: e.root(E.let_("hours", shipment.hours.add(to_go), E.variable("hours").le(72))))
with store.transaction():
    trip_left = Expression(lambda e: e.root(Partials.OfAny(total, {"shipment": s1})))
    decided = Expression(lambda e: e.root(Partials.OfAny(total, {"shipment": s1, "to_go": 50})))
print(show(total))
print(show(trip_left))
print(show(decided))

let hours = add(get(shipment, 'hours'), to_go) in le(hours, 72)
let hours = add(30, to_go) in le(hours, 72)
False


With `to_go` unknown, the let stays, now binding `add(30, to_go)`. With both known, the whole constraint is a value.

## Why it works this way

- **The residual is the question that's left.** Hand it to whoever knows the rest, such as the routing service that
  knows the hours to go, and they can answer it without knowing anything about shipments.
- **Specializing is evaluating.** A template's parameters are variables like any other, so specializing it needs no
  new mechanism.
- **Closed parts are evaluated whole.** A subexpression whose variables are all known is replaced by its value when it
  has one. When its value is unknown, as S3's temperature is, it stays as written: the residual still says what to ask.

Partial evaluation, in full: [CAP-099: partial evaluation](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-099.md#partials).

<!-- nav -->
[← 11 · Constraints over collections](11_Constraints_Over_Collections.ipynb) · [Home](../../README.md) · [13 · Analyzing and rewriting constraints →](13_Analyzing_And_Rewriting_Constraints.ipynb)